# 📝 Homework Day 1: Data Engineering Pipeline
## Agentic RAG: From Zero to Hero

---

### 📋 Instructions

1. **Complete this by yourself** — do not use AI to help write code
2. **No copying** — each person's data will be **different** (generated from the student ID)
3. **Submit this notebook** together with the executed results (.ipynb)
4. **Score**: 10 points

> ⚠️ **The system will detect plagiarism** using hash values, embeddings, and scores that must match each student's ID

## 📦 Install Dependencies

In [ ]:
%%time
!pip install -q sentence-transformers qdrant-client langchain-text-splitters rank_bm25 pythainlp scikit-learn

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ Installation complete!')

## 🎓 Fill in Student Information

In [ ]:
# ─── Fill in your information ───
STUDENT_NAME = ''   # e.g. 'Somchai Jaidee'
STUDENT_ID   = ''   # e.g. '6512345678'
PHONE        = ''   # e.g. '081-234-5678'
LINE_ID      = ''   # e.g. 'somchai.j'

# ─── Validation (do not edit) ───
assert len(STUDENT_ID) >= 5, '❌ Please enter your student ID!'
assert len(STUDENT_NAME) >= 2, '❌ Please enter your full name!'

print(f'✅ Full name: {STUDENT_NAME}')
print(f'✅ Student ID: {STUDENT_ID}')
print(f'📱 Phone number: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 Step 2: Create Your Personalized Dataset (Do not edit this cell)

In [ ]:
%%time
# ===== Do not edit this cell =====
# Create a personalized dataset from the student ID

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

# Main content — shuffle order and select content based on the student ID
all_paragraphs = [
    'Machine Learning is a subfield of artificial intelligence that focuses on developing algorithms that can learn from data and automatically improve performance without requiring explicit programming for every case',
    'Deep Learning is a technique within Machine Learning that uses multi-layer neural networks to process complex data such as image recognition, language translation, and text generation',
    'Natural Language Processing or NLP is the field concerned with enabling computers to understand, interpret, and generate human language, including sentiment analysis, summarization, and question answering',
    'Retrieval Augmented Generation or RAG is a technique that combines information retrieval with LLM text generation in order to produce accurate, up-to-date, and source-grounded answers',
    'Vector Database is a database designed to store and search data in the form of embedding vectors, enabling fast retrieval of semantically similar information',
    'Text Embedding is the process of converting text into a set of numbers or vectors that represent its semantic meaning, allowing computers to compare the similarity between texts',
    'Transformer is a neural network architecture that uses the Attention mechanism to process sequential data and serves as the foundation of large language models such as GPT, BERT, and Gemini',
    'Prompt Engineering is the discipline of designing instructions or prompts for LLMs in order to obtain the desired output. A well-written prompt can greatly improve answer quality and accuracy',
    'Fine-tuning is the process of adapting a pre-trained model with additional domain-specific data so that it performs better on specialized tasks such as medical document analysis',
    'Tokenization is the process of splitting text into smaller units called tokens, which may be words, subwords, or characters. For Thai, word segmentation is more complex because there are no spaces between words',
    'Chunking is the process of splitting a long document into smaller parts suitable for embedding generation. There are many methods such as fixed-size, recursive, and semantic chunking',
    'Cosine Similarity is a method for measuring similarity between two vectors by looking at the angle between them. A value of 1 means the same direction, and 0 means orthogonal. It is commonly used in NLP and information retrieval',
]

# Shuffle order based on the student's seed
random.shuffle(all_paragraphs)

# Select 8 paragraphs + create 1 duplicate
selected = all_paragraphs[:8]
duplicate_idx = random.randint(0, 5)
selected.append(selected[duplicate_idx])  # duplicated paragraph

# Save as files
os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ Personalized data for {STUDENT_ID} has been created successfully!')
print(f'📁 Number of files: {len(selected)} files (1 duplicate file included)')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} characters)')

---
## 🎯 Task: Build a Data Engineering Pipeline

Build a **RAG Data Pipeline** from your personalized dataset by following the steps below.
Each step has **answers that must be reported** — these values will be different for each student.

---

### Step 1: Detect Duplicates with MD5 (2 points)

- Compute the MD5 hash of every file in `homework_data/`
- Find the duplicate files
- Remove the duplicate file(s) (keep the first one)

**📝 Report:**
1. Which pair of files is duplicated?
2. What is the MD5 hash of the duplicate file?
3. How many files remain after removal?

In [ ]:
# Step 1: Write your duplicate detection code here

# 💡 Hint:
#   1. import hashlib
#   2. Create a function compute_md5(filepath) that reads a file and returns the hash
#   3. Loop through every file in 'homework_data/' and store hashes in a dict
#   4. If a hash repeats → that file is a duplicate



# ✅ Self-check (run after finishing your code)
# assert dup_hash is not None, '❌ You have not found the duplicate file yet'
# assert files_remaining >= 7, '❌ The number of files after deletion should be greater than 7'
# print(f'✅ Step 1 passed: duplicate files={dup_files}, MD5={dup_hash}')

### Step 2: Chunking (2 points)

- Combine the text from the remaining files (excluding duplicates)
- Chunk using `RecursiveCharacterTextSplitter` with `chunk_size=150`, `chunk_overlap=30`

**📝 Report:**
1. How many chunks are there in total?
2. What is the content of chunk 1? (copy and paste it)
3. How many characters are in the shortest chunk?

In [ ]:
# Step 2: Write your chunking code here

# 💡 Hint:
#   1. from langchain_text_splitters import RecursiveCharacterTextSplitter
#   2. Combine content from non-duplicate files
#   3. Create a splitter with chunk_size=150, chunk_overlap=30
#   4. Use splitter.split_text(all_text)



# ✅ Self-check
# assert len(chunks) > 0, '❌ You have not chunked the text yet'
# assert len(chunks[0]) <= 150, '❌ The chunk is too large'
# print(f'✅ Step 2 passed: {len(chunks)} chunks, chunk 1 = {len(chunks[0])} chars')

### Step 3: Embedding + Similarity (3 points)

- Create embeddings using `intfloat/multilingual-e5-large`
- Compute **Cosine Similarity** between the query and every chunk
- Query: `'query: techniques for retrieving semantically similar information'`

**📝 Report:**
1. Which chunk has the highest similarity? (specify the chunk number and score to 4 decimal places)
2. Which chunk has the lowest similarity? (specify the chunk number and score to 4 decimal places)
3. Based on the result, why is that chunk the most similar? (Explain in your own words in 2–3 sentences)

In [ ]:
# Step 3: Write your embedding + similarity code here
# Don't forget to add the 'query: ' and 'passage: ' prefixes as taught in class

# 💡 Hint:
#   1. from sentence_transformers import SentenceTransformer
#   2. model = SentenceTransformer('intfloat/multilingual-e5-large')
#   3. query = 'query: techniques for retrieving semantically similar information'
#   4. passages = ['passage: ' + c for c in chunks]
#   5. query_emb = model.encode(query)
#   6. passage_embs = model.encode(passages)
#   7. Use cosine_similarity() from sklearn



# ✅ Self-check
# assert best_score > 0.5, '❌ similarity score is too low, check the prefixes'
# print(f'✅ Step 3 passed: best chunk={best_idx}, score={best_score:.4f}')

### Step 4: Qdrant Retrieval (3 points)

- Create a Qdrant client (in-memory) + a collection named `f'hw_{STUDENT_ID}'`
- Upsert every chunk + embedding into Qdrant
- Search with the query: `'query: splitting text into smaller parts'`
- Use `top_k=3`

**📝 Report:**
1. For the Top-3 results, what is the score of each rank? (4 decimal places)
2. What is the top-ranked result about?
3. Write a summary: if this system were to be used in a real-world setting, what should be improved? (Explain in 3–5 sentences)

In [ ]:
# Step 4: Write your Qdrant retrieval code here

# 💡 Hint:
#   1. from qdrant_client import QdrantClient, models
#   2. qdrant = QdrantClient(':memory:')
#   3. Create a collection named f'hw_{STUDENT_ID}'
#   4. Upsert every chunk + embedding into Qdrant
#   5. query = 'query: splitting text into smaller parts'
#   6. qdrant.query_points(..., limit=3)



# ✅ Self-check
# assert len(results) == 3, '❌ You should get top_k=3 results'
# print(f'✅ Step 4 passed: Top-3 scores={[r.score for r in results]}')

## 📊 Grading Criteria

| Step | Score | Criteria |
|---------|:-----:|------|
| 1. Find Duplicates | 2 | `find_duplicates` function works correctly, results are correct |
| 2. Chunking | 3 | chunk_size/overlap are correct, methods are compared |
| 3. Embedding + Search | 3 | embedding works, search works, results are reasonable |
| 4. Result Analysis | 2 | explains output quality, compares parameters |
| **Total** | **10** | |

---
## ✅ Verify Your Answers

Run the cell below to generate a **Verification Code** for submission

In [ ]:
# ===== Do not edit this cell =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_day1_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 Full name: {STUDENT_NAME}')
print(f'🎓 Student ID: {STUDENT_ID}')
print(f'📱 Phone number: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 Submit before: _________________ (set by instructor)')
print('=' * 50)
print()
print('📋 Checklist before submission:')
print('  [ ] Filled in all personal information completely')
print('  [ ] Step 1: Identified duplicate file(s) + MD5 hash')
print('  [ ] Step 2: Specified the number of chunks + content of chunk 1')
print('  [ ] Step 3: Specified the most similar/least similar chunk + explanation')
print('  [ ] Step 4: Specified Top-3 scores + improvement summary')
print('  [ ] All cells have been run and show outputs')